# TP8 : atelier incidents en binômes

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 15:35-16:10, après la démo du TP (15:30-15:35) |
| Compétences | C9 Listeners, TLS et SASL ; C13 Diagnostic ; C16 Méthode de diagnostic d'incident |
| Slides de rappel | S70 (méthode), S71 (arbre de décision et messages d'erreur), S72 (boîte à outils) |
| Fiche à remplir | `fiche-diagnostic.md`, une copie par ticket (clic droit sur le fichier, *Duplicate*) |
| Version Markdown | `enonce.md`, mêmes tickets en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez d'abord la cellule d'outils, puis l'étape 7.1. Après un redémarrage du noyau, relancez la cellule d'outils et la cellule de 7.1.
- Les sections 7.2 à 7.4 sont les trois tickets. Traitez-les dans l'ordre de vos tours (tableau de 7.1), pas dans l'ordre du notebook.
- Dans chaque ticket : une **cellule de reproduction** fournie, une **zone `# TODO` de diagnostic** (ajoutez autant de cellules que nécessaire, bouton `+`), une **zone `# TODO` de correction**, puis une **vérification**, qui contrôle l'objectif de résolution du ticket.
- **Règle du jeu** : ne lisez pas `scripts/incident.sh` et n'utilisez pas ses commandes `status` : elles donnent la réponse. Le diagnostic se fait avec la boîte à outils ci-dessous, comme en production.
- Les messages qui commencent par `%3|` ou `%4|` viennent de la bibliothèque cliente (librdkafka) : ce sont des erreurs et avertissements du client Python. Ils apparaissent sous les cellules qui produisent ou consomment des messages, comme les cellules de reproduction. Lisez-les : ce sont des indices.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import subprocess
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import os
import time
from collections import Counter

import requests
from confluent_kafka import OFFSET_BEGINNING, Consumer, KafkaError, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, OffsetSpec

# Boite a outils du diagnostic (section 8.3 de la version Markdown)


def promql(requete, etiquette=None):
    """Requete PromQL instantanee : {valeur de l'etiquette: valeur}."""
    resultats = requests.get(PROMETHEUS + "/api/v1/query", params={"query": requete}, timeout=10).json()["data"]["result"]
    return {r["metric"].get(etiquette, "") if etiquette else "": float(r["value"][1]) for r in resultats}


def alertes():
    """Alertes actives de Prometheus : ["nom:etat", ...], avec (xN) quand plusieurs noeuds sont concernes."""
    donnees = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=10).json()["data"]["alerts"]
    compte = Counter("%s:%s" % (a["labels"]["alertname"], a["state"]) for a in donnees)
    return sorted(cle + (" (x%d)" % n if n > 1 else "") for cle, n in compte.items())


def sous_min_isr():
    """A ecrire a l'etape 7.1."""
    return None


def avec_reprise(fonction, essais=8):
    """Appelle fonction() ; en cas d'erreur Kafka, nouvel essai (admin cree un client neuf a chaque appel).

    describe_cluster() et describe_configs() partent vers le broker que les metadonnees designent comme
    controleur, tire au hasard en KRaft a chaque reponse : s'il est injoignable, la requete echoue au bout
    du delai, et l'essai suivant a de bonnes chances de viser un autre broker.
    """
    admin = AdminClient(CONF)
    for essai in range(essais):
        try:
            return fonction()
        except KafkaException as e:
            if essai == essais - 1:
                raise
            print("[%s : %s, nouvel essai avec un client neuf]" % (fonction.__name__, e.args[0].name()))


def bilan(afficher=True):
    """Etat general, du quorum aux alertes : la reference « tout va bien » de 7.1 sert de comparaison."""
    admin = AdminClient(CONF)
    statut = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status 2>&1
    statut = statut.n
    metadonnees = admin.list_topics(timeout=10)
    partitions = [p for t in metadonnees.topics.values() for p in t.partitions.values()]
    resultat = {
        "quorum": "leader %s, lag max %s" % (statut.split("LeaderId:")[1].split()[0],
                                             statut.split("MaxFollowerLag:")[1].split()[0]),
        # Noms annonces, lus dans les metadonnees : elles viennent de n'importe quel broker joignable
        "brokers": {b.id: b.host for b in sorted(metadonnees.brokers.values(), key=lambda b: b.id)},
        "urp": sum(1 for p in partitions if len(p.isrs) < len(p.replicas)),
        "sous_min_isr": avec_reprise(sous_min_isr),
        "indisponibles": sum(1 for p in partitions if p.leader == -1),
        "alertes": alertes(),
    }
    if afficher:
        print("== %s (heure UTC du conteneur)" % time.strftime("%H:%M:%S"))
        for cle, valeur in resultat.items():
            print("%-14s %s" % (cle, valeur))
    return resultat


def decrire(topic):
    """Leader, repliques et ISR de chaque partition d'un topic."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions
    tableau([(p.id, p.leader, p.replicas, p.isrs) for p in sorted(partitions.values(), key=lambda p: p.id)],
            ["Partition", "Leader", "Repliques", "ISR"])


def config_topic(topic):
    """Reglages propres a un topic (source DYNAMIC_TOPIC_CONFIG) : {nom: valeur}."""
    admin = AdminClient(CONF)
    futur = list(admin.describe_configs([ConfigResource("topic", topic)]).values())[0]
    return {nom: e.value for nom, e in futur.result().items() if e.source == ConfigSource.DYNAMIC_TOPIC_CONFIG.value}


def traces(noeud, motif, depuis="3m", lignes=10):
    """Dernieres lignes des traces d'un noeud qui correspondent au motif (expression reguliere etendue)."""
    sortie = !docker compose logs kafka-{noeud} --since {depuis} --no-log-prefix 2>&1 | grep -E '{motif}' | tail -{lignes} | cut -c1-250
    return sortie.n


def tailles_par_broker():
    """Donnees stockees par noeud, en Mo (metrique kafka_log_log_size collectee par Prometheus)."""
    return dict(sorted((n, round(v / 1048576)) for n, v in
                       promql("sum by (instance) (kafka_log_log_size)", "instance").items()))


def tailles_par_topic():
    """Donnees stockees par topic, toutes repliques confondues, en Mo, du plus gros au plus petit."""
    return dict(sorted(((t, round(v / 1048576, 1)) for t, v in
                        promql("sum by (topic) (kafka_log_log_size)", "topic").items()), key=lambda x: -x[1]))


def suivre(nombre, pas=15, jusqua=None):
    """Releve, toutes les pas secondes, la taille par noeud et les alertes ; s'arrete des que jusqua(alertes) est vrai."""
    debut = time.time()
    for _ in range(nombre):
        time.sleep(pas)
        actives = alertes()
        print("t+%3ds  %s  alertes: %s" % (time.time() - debut, "  ".join("%s=%dMo" % kv for kv in
                                                                          tailles_par_broker().items()), " ".join(actives)))
        if jusqua and jusqua(actives):
            break


def lire_tout(topic, delai=10):
    """Toutes les cles et valeurs d'un topic, depuis le debut (lecture par assign, sans groupe)."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions
    consommateur = Consumer({"bootstrap.servers": BS, "group.id": "tp8-lecture", "enable.auto.commit": False,
                             "enable.partition.eof": True, "log_level": 4})
    consommateur.assign([TopicPartition(topic, p, OFFSET_BEGINNING) for p in partitions])
    messages, finies, limite = [], set(), time.time() + delai
    while len(finies) < len(partitions) and time.time() < limite:
        msg = consommateur.poll(1)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                finies.add(msg.partition())
            continue
        messages.append(((msg.key() or b"").decode(), msg.value().decode()))
    consommateur.close()
    return messages


def messages_presents(topic):
    """Nombre de messages encore presents dans un topic : fin moins debut, sur toutes ses partitions."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions

    def offsets(spec):
        demandes = {TopicPartition(topic, p): spec for p in partitions}
        return {tp.partition: f.result().offset for tp, f in admin.list_offsets(demandes).items()}

    debut, fin = offsets(OffsetSpec.earliest()), offsets(OffsetSpec.latest())
    return sum(fin[p] - debut[p] for p in partitions)


print("Prometheus joignable :", requests.get(PROMETHEUS + "/-/ready", timeout=5).status_code == 200)

## 1. Objectifs

À la fin du TP, vous savez :

- qualifier un incident à partir d'un ticket et d'un message d'erreur ;
- observer le cluster du général au particulier, avant d'agir ;
- formuler une hypothèse et la vérifier par une commande ;
- corriger par l'action la moins risquée, puis prouver le retour à la normale ;
- rédiger une fiche d'incident exploitable par une autre équipe.

## 2. Durée

35 minutes, après 5 minutes de démonstration.

| Étape | Durée |
|---|---|
| 7.1 Préparer le binôme et le lab | 5 min |
| Tour 1 | 10 min |
| Tour 2 | 10 min |
| Tour 3 | 10 min |

Dans chaque tour : 1 minute d'injection, 7 minutes de diagnostic et de correction, 1 minute de validation, 1 minute d'échange entre les deux membres du binôme.

## 3. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh` sans échec. Les incidents utilisent le générateur de charge et Prometheus.
- TP5 et TP6 terminés : listener SECURE, comptes SCRAM, requêtes Prometheus et alertes.
- Une copie de `fiche-diagnostic.md` par ticket.
- Notions : slides S70 à S72, et S71 en particulier pour les messages d'erreur.

## 4. Architecture utilisée

```
  Binôme : stagiaire A (lab A) et stagiaire B (lab B)

  Tour n :  B injecte un incident sur le lab A  --->  A diagnostique et corrige sur son lab
            A injecte un incident sur le lab B  --->  B diagnostique et corrige sur son lab
            (en parallèle, puis échange de fiches)

  Lab : kafka-1..3 (broker et contrôleur), listeners INTERNAL :19092 et SECURE :19094 (SASL_SSL),
        loadgen 100 messages/s, Prometheus :9090 et ses alertes, Grafana :3000
```

L'injecteur connaît l'incident et joue l'équipe qui a ouvert le ticket. Il répond aux questions sur le contexte, jamais sur la cause.

| Lab | Production |
|---|---|
| L'incident est injecté par un script, et réparé par le même script en fin de tour | L'incident n'est pas connu ; la correction passe par la gestion de configuration et le processus de changement |
| Seuil disque de l'alerte `KafkaBrokerLogSizeHigh` : 250 Mo par nœud | Seuil en pourcentage du disque, mesuré par l'exporteur du système (node_exporter) |
| Vérification de la rétention toutes les 30 s | 5 minutes par défaut (`log.retention.check.interval.ms`) |

## 5. Contexte métier

Fin d'après-midi chez NovaShop. Trois tickets arrivent à l'équipe d'exploitation Kafka, dont vous assurez l'astreinte. Chacun doit être traité avec méthode : une correction hasardeuse sur le cluster de production coûterait plus cher que l'incident lui-même.

## 6. Énoncé

Chaque stagiaire traite les trois tickets, sur son propre lab, dans l'ordre fixé par le tableau de 7.1. Pour chacun : diagnostic, correction, validation et fiche remplie.

| Ticket | Équipe | Texte du ticket | Objectif de résolution |
|---|---|---|---|
| **Logistique** | Logistique | « Notre nouveau service `shipping-service` doit publier ses événements d'expédition dans le topic `shipments`, sur le listener sécurisé. On nous a créé le compte `shipping` et on a livré notre fichier de configuration. L'envoi échoue. Notre script de test : `/tmp/incident4/send-shipment.sh` dans `tools`. » | Le script se termine sans erreur et l'événement est lu dans `shipments`. |
| **Paiements** | Paiements | « Depuis une intervention sur `kafka-3`, une partie des paiements n'est plus enregistrée : certains envois expirent, d'autres passent. Les alertes de réplication se sont allumées. » | Tous les envois passent ; URP et partitions sous min ISR à 0 ; alertes levées. |
| **Stockage** | Supervision | « L'alerte `KafkaBrokerLogSizeHigh` s'est déclenchée sur les 3 nœuds. L'équipe Data a lancé une collecte de `clickstream` cet après-midi. Sa politique : garder au plus 1 heure et 5 Mo par partition. » | Volume revenu sous le seuil, limite en place contre la récidive, sans supprimer le topic ; alerte levée. |

## 7. Étapes

### 7.1 Préparer le binôme et le lab

Fixez l'ordre des tickets :

| Tour | Lab de A (injecté par B) | Lab de B (injecté par A) |
|---|---|---|
| 1 | Paiements | Logistique |
| 2 | Stockage | Paiements |
| 3 | Logistique | Stockage |

Correspondance, **réservée à l'injecteur** : Logistique = 4, Paiements = 5, Stockage = 6. Injection, dans un terminal du poste de votre partenaire, dossier `lab/` : `scripts/incident.sh N inject`.

La fonction `bilan()` de la boîte à outils donne l'état général. Il lui manque un indicateur : le nombre de partitions **sous min ISR**, celles qui refusent les écritures `acks=all`.

**Exemple en ligne de commande** : les trois compteurs de partitions dégradées.

In [ ]:
!docker exec kafka-tools bash -c 'for option in under-replicated-partitions under-min-isr-partitions unavailable-partitions; do echo "$$option : $$(kafka-topics.sh --bootstrap-server $BS --describe --$$option | grep -c Partition)"; done'

In [ ]:
# TODO 7.1 : completer la boite a outils, puis enregistrer le bilan de reference
# 1. Ecrivez sous_min_isr() : nombre de partitions dont l'ISR compte moins de repliques que le min.insync.replicas
#    de leur topic. Lisez min.insync.replicas de tous les topics en un seul appel :
#    admin.describe_configs([ConfigResource("topic", t) for t in topics], request_timeout=5)
#    renvoie {ressource: futur} ;
#    ressource.name est le nom du topic, futur.result()["min.insync.replicas"].value sa valeur (texte).
# 2. reference = bilan() : c'est votre etat « tout va bien », a comparer a chaque fin de tour.
def sous_min_isr():
    return None


reference = {}

In [ ]:
# Verification 7.1
assert reference.get("sous_min_isr") == 0, "A revoir : sous_min_isr() ecrite : aucune partition sous min ISR"
print("OK  sous_min_isr() ecrite : aucune partition sous min ISR")
assert reference.get("brokers") == {1: "kafka-1", 2: "kafka-2", 3: "kafka-3"}, "A revoir : 3 brokers, annonces sous leur nom"
print("OK  3 brokers, annonces sous leur nom")
assert reference.get("urp") == 0 and reference.get("indisponibles") == 0, "A revoir : aucune partition degradee"
print("OK  aucune partition degradee")
assert reference.get("alertes") == [], "A revoir : aucune alerte active : signalez au formateur toute alerte deja presente"
print("OK  aucune alerte active : signalez au formateur toute alerte deja presente")

**Résultat attendu** : un leader de quorum, les brokers `kafka-1`, `kafka-2` et `kafka-3`, les trois compteurs à 0 et aucune alerte. Juste après le TP7, une alerte `pending` (`KafkaFencedBrokers`, `KafkaUnderMinIsrPartitions`) peut subsister une trentaine de secondes après l'arrêt de `kafka-4` : attendez, puis relancez `bilan()`. Si une alerte reste active, signalez-le au formateur avant de commencer.

### 7.2 Ticket Logistique

> « Notre nouveau service `shipping-service` doit publier ses événements d'expédition dans le topic `shipments`, sur le listener sécurisé. On nous a créé le compte `shipping` et on a livré notre fichier de configuration. L'envoi échoue. Notre script de test : `/tmp/incident4/send-shipment.sh` dans `tools`. »

**Reproduction** : le script de test de l'équipe, lancé dans `tools`. La fonction retire de sa sortie les lignes `at ...` des piles d'appels Java, puis compte les lignes `ERROR`.

Le code retour vaut 0 même quand l'envoi échoue : `kafka-console-producer.sh` ne signale pas un échec d'envoi par son code retour. Ce sont les lignes `ERROR` qui comptent.

In [ ]:
def reproduire_logistique():
    commande = "/tmp/incident4/send-shipment.sh 2>&1 | grep -v '^[[:space:]]*at '; exit ${PIPESTATUS[0]}"
    resultat = subprocess.run(["docker", "exec", "kafka-tools", "bash", "-c", commande], capture_output=True, text=True)
    sortie = resultat.stdout.rstrip("\n")
    print("code retour : %d, lignes ERROR : %d" % (resultat.returncode, sortie.count(" ERROR ")))
    return sortie

In [ ]:
# TODO 7.2 a : diagnostic du ticket Logistique
# Reproduisez, observez du general au particulier, puis verifiez chaque hypothese par une commande.
# Ajoutez une cellule par hypothese. Notez dans la fiche chaque commande et son constat.
# Boite a outils : bilan(), decrire(topic), config_topic(topic), traces(noeud, motif), alertes(), promql(...),
# des cellules `!docker exec kafka-tools ...` et `!docker compose ...` ; commandes de la section 8.3 de enonce.md.
reproduire_logistique()

In [ ]:
# TODO 7.2 b : correction du ticket Logistique
# Appliquez la correction la moins risquee (commande `!docker exec kafka-tools ...`, `!docker compose ...` ou appel `admin`), puis relancez reproduire_logistique().

In [ ]:
# Verification 7.2
essai = reproduire_logistique()
assert "ERROR" not in essai, "A revoir : le script de test se termine sans erreur"
print("OK  le script de test se termine sans erreur")
expeditions = lire_tout("shipments")
assert any("colissimo" in valeur for _, valeur in expeditions), "A revoir : evenement d'expedition lu dans shipments (%d message(s))" % len(expeditions)
print("OK  evenement d'expedition lu dans shipments (%d message(s))" % len(expeditions))
print(expeditions[-1])

**Résultat attendu** :

- le script se termine sans aucune ligne `ERROR`, avec le code retour 0 ;
- la lecture de `shipments` depuis le début affiche l'événement `SHIP-<horodatage>` avec `"carrier":"colissimo"` ;
- les traces des brokers ne montrent plus de nouvel échec d'authentification pour ce client.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 4 repair` dans un terminal de votre poste. Il réapplique l'état de référence, même si votre correction est déjà la bonne, et envoie un événement de plus. Comparez ensuite le bilan à la référence de 7.1.

In [ ]:
bilan()

### 7.3 Ticket Paiements

> « Depuis une intervention sur `kafka-3`, une partie des paiements n'est plus enregistrée : certains envois expirent, d'autres passent. Les alertes de réplication se sont allumées. »

Attendez 30 s après le `[ OK ]` de l'injection avant la première observation.

**Reproduction** : 6 paiements, un par clé, chacun avec 20 s pour être confirmé. Le partitionneur `murmur2_random` place chaque clé dans la même partition que le client Java de la version Markdown.

```bash
# [tools] equivalent en ligne de commande
for k in PAY-9001 PAY-9002 PAY-9003 PAY-9004 PAY-9005 PAY-9006; do echo "$k|{\"amount\":42}"; done \
  | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
      --reader-property parse.key=true --reader-property 'key.separator=|' \
      --command-property delivery.timeout.ms=20000 --command-property request.timeout.ms=10000
```

In [ ]:
def reproduire_paiements():
    resultats = {}

    def rapport(err, msg):
        resultats[msg.key().decode()] = ("payments-%d" % msg.partition(), err.name() if err else "OK")

    producteur = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4,
                           "delivery.timeout.ms": 20000, "request.timeout.ms": 10000})
    for numero in range(9001, 9007):
        producteur.produce("payments", key="PAY-%d" % numero, value='{"amount":42}', on_delivery=rapport)
    producteur.flush(30)
    tableau([(cle,) + resultat for cle, resultat in sorted(resultats.items())], ["Cle", "Partition", "Resultat"])
    return resultats

In [ ]:
# TODO 7.3 a : diagnostic du ticket Paiements
# Quels envois passent, lesquels echouent, sur quelles partitions ? Puis observez du general au particulier.
# Ajoutez une cellule par hypothese. Notez dans la fiche chaque commande et son constat.
reproduire_paiements()

In [ ]:
# TODO 7.3 b : correction du ticket Paiements
# Appliquez la correction la moins risquee, sur un seul noeud, puis attendez son retour (sain, URP a 0).
# Les alertes de replication se levent 1 a 2 minutes apres la correction.

In [ ]:
# Verification 7.3
resultats = reproduire_paiements()
assert len(resultats) == 6 and all(r == "OK" for _, r in resultats.values()), "A revoir : les 6 paiements passent"
print("OK  les 6 paiements passent")
etat = bilan()
assert etat["brokers"] == {1: "kafka-1", 2: "kafka-2", 3: "kafka-3"}, "A revoir : brokers annonces sous leur nom"
print("OK  brokers annonces sous leur nom")
assert etat["urp"] == 0 and etat["sous_min_isr"] == 0 and etat["indisponibles"] == 0, "A revoir : URP, sous min ISR et indisponibles a 0"
print("OK  URP, sous min ISR et indisponibles a 0")
assert not [a for a in etat["alertes"] if a.startswith(("KafkaUnderReplicated", "KafkaUnderMinIsr"))], "A revoir : alertes de replication levees (sinon, relancez cette cellule dans 1 minute)"
print("OK  alertes de replication levees (sinon, relancez cette cellule dans 1 minute)")

**Résultat attendu** :

- les 6 envois passent ;
- `bilan()` : les brokers sont annoncés sous les noms `kafka-1`, `kafka-2`, `kafka-3` ; URP, sous min ISR et indisponibles à 0 ;
- les alertes de réplication sont levées dans les 1 à 2 minutes qui suivent la correction.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 5 repair` dans un terminal de votre poste. Il redémarre `kafka-3` une seconde fois, environ 15 s. Attendez que l'URP revienne à 0, puis comparez le bilan à la référence de 7.1.

In [ ]:
bilan()

### 7.4 Ticket Stockage

> « L'alerte `KafkaBrokerLogSizeHigh` s'est déclenchée sur les 3 nœuds. L'équipe Data a lancé une collecte de `clickstream` cet après-midi. Sa politique : garder au plus 1 heure et 5 Mo par partition. »

Attendez l'alerte : 1 min 30 à 2 min après l'injection, selon le volume déjà présent sur votre lab. La fonction `suivre()` de la boîte à outils relève la taille par nœud et les alertes toutes les 15 s.

**Reproduction** : les alertes, et la taille des données par nœud (panneau « Donnees stockees par broker » de Grafana).

In [ ]:
def reproduire_stockage():
    print("alertes :", alertes())
    print("donnees par noeud (Mo) :", tailles_par_broker())

In [ ]:
# TODO 7.4 a : diagnostic du ticket Stockage
# Quel topic grossit ? Pourquoi ? Ajoutez une cellule par hypothese. Notez dans la fiche chaque commande et son constat.
reproduire_stockage()

In [ ]:
# TODO 7.4 b : correction du ticket Stockage
# Appliquez une correction conforme a la politique de l'equipe Data, sans supprimer le topic ni ses fichiers.
# Puis suivez le retour sous le seuil : suivre(8, jusqua=lambda actives: not any(...)).

In [ ]:
# Verification 7.4
reglages = config_topic("clickstream")
assert 0 < int(reglages.get("retention.ms", -1)) <= 3600000 and 0 < int(reglages.get("retention.bytes", -1)) <= 5242880, "A revoir : clickstream limite a 1 h et 5 Mo par partition"
print("OK  clickstream limite a 1 h et 5 Mo par partition")
tailles = tailles_par_broker()
assert tailles and max(tailles.values()) < 250, "A revoir : moins de 250 Mo par noeud : %s" % tailles
print("OK  moins de 250 Mo par noeud : %s" % tailles)
assert not [a for a in alertes() if a.startswith("KafkaBrokerLogSizeHigh")], "A revoir : alerte KafkaBrokerLogSizeHigh levee (sinon, relancez cette cellule dans 30 s)"
print("OK  alerte KafkaBrokerLogSizeHigh levee (sinon, relancez cette cellule dans 30 s)")
assert messages_presents("clickstream") > 0, "A revoir : topic present, avec %d messages recents" % messages_presents("clickstream")
print("OK  topic present, avec %d messages recents" % messages_presents("clickstream"))

**Résultat attendu** :

- la configuration de `clickstream` porte une limite de durée et de taille conforme à la politique de l'équipe Data ;
- moins de 250 Mo par nœud ;
- `KafkaBrokerLogSizeHigh` absente des alertes, environ 1 minute après la correction ;
- le topic existe toujours, avec ses données récentes.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 6 repair` dans un terminal de votre poste : il réapplique la même rétention. Comparez ensuite le bilan à la référence de 7.1.

In [ ]:
bilan()

## Bilan de l'exercice

À comparer au bilan de référence de 7.1. Les trois fiches remplies, relues et signées par l'injecteur, complètent ce bilan.

In [ ]:
final = bilan(afficher=False)
sauvegarde_incident5 = os.path.exists(os.path.join(LAB_DIR, "config/kafka/kafka-3/server.properties.before-incident5"))
tableau([(cle, final[cle], reference.get(cle, "a completer en 7.1")) for cle in ("brokers", "urp", "sous_min_isr",
                                                                                  "indisponibles", "alertes")]
        + [("sauvegarde de l'incident 5 presente", sauvegarde_incident5, False)],
       ["Controle", "Fin du TP", "Reference (7.1)"])

## 11. Questions de réflexion

1. Ticket Paiements : pourquoi certains envois passent-ils alors que d'autres expirent ?
2. Ticket Paiements : `kafka-3` est `healthy` pour Docker et `unfenced` pour le contrôleur. Pourquoi est-il pourtant inutilisable pour les clients ?
3. Ticket Paiements : pourquoi l'alerte `KafkaNodeDown` ne s'est-elle pas déclenchée ?
4. Ticket Logistique : pourquoi le client a-t-il rencontré deux erreurs successives ? Que disaient les traces du broker dans chaque cas ?
5. Ticket Logistique : fallait-il corriger le client ou le broker ? Pourquoi ?
6. Ticket Stockage : pourquoi une limite de durée d'une heure, seule, n'aurait-elle rien libéré ?
7. Ticket Stockage : pourquoi l'espace n'est-il pas libéré à l'instant même de la correction ?
8. Pour chaque ticket, quelle mesure de prévention proposez-vous ?

## 12. Erreurs fréquentes

| Symptôme | Cause | Correction |
|---|---|---|
| Diagnostic trop rapide, cause « devinée » | Lecture de `scripts/incident.sh` ou de ses commandes `status` | Repartir des symptômes ; la fiche doit montrer la démarche |
| Les indices ont disparu | Redémarrage d'un nœud avant d'avoir observé | Observer d'abord ; redémarrer est une correction, pas un diagnostic |
| `bilan()` affiche `sous_min_isr None` | Étape 7.1 non faite, ou noyau redémarré depuis | Relancer la cellule d'outils, puis la cellule de 7.1 |
| Des lignes `%3|...` apparaissent sous une cellule | Journal d'un producteur ou d'un consommateur Python (librdkafka) qui ne joint pas un broker | Ce n'est pas une erreur de votre code : lisez-les, ce sont des indices |
| Paiements : `KafkaException ... _TIMED_OUT ... Failed while waiting for controller` dans une cellule de diagnostic | `describe_cluster()` et `describe_configs()` visent le broker que les métadonnées désignent comme contrôleur, tiré au hasard : quand c'est `kafka-3`, injoignable sous son nom annoncé, la requête expire | `bilan()` refait l'essai et l'affiche ; ailleurs, relancer la cellule. C'est un symptôme de l'incident, pas une erreur de code |
| Paiements : les symptômes reviennent après un redémarrage de `kafka-3` | Le fichier de configuration fautif n'a pas été corrigé | Comparer avec `config/kafka/pristine/`, corriger, puis redémarrer |
| Logistique : code retour 0, et pourtant l'envoi a échoué | `kafka-console-producer.sh` ne signale pas un échec d'envoi par son code retour | Lire les lignes `ERROR`, puis chercher l'événement dans `shipments` |
| Logistique : une nouvelle erreur apparaît après la première correction | Le fichier client contient plusieurs erreurs | C'est un progrès : lire le nouveau message et continuer |
| Logistique : modification du broker pour accepter le client | Mauvaise cible : le listener est conforme à la politique de sécurité | Corriger le fichier du client |
| Stockage : topic supprimé, ou fichiers effacés dans `/var/lib/kafka` | Correction destructrice | Interdit ; on agit par `retention.ms` et `retention.bytes` |
| Stockage : rien n'est libéré après la correction | Limite de durée seule, ou attente trop courte | Ajouter la limite de taille ; la suppression a lieu toutes les 30 s dans le lab |
| `Incident 5 deja injecte (sauvegarde presente).` | `repair` oublié au tour précédent | `scripts/incident.sh 5 repair`, puis nouvelle injection |
| `ConnectionError` vers `prometheus` | Lab en mode `core` | `scripts/start.sh extended` |

## 13. Solution formateur

Voir `corrige.ipynb` (et `corrige.md` pour la version en ligne de commande).

## 14. Variante avancée

1. **Trois incidents de plus.** Les binômes rapides traitent les incidents 1, 2 et 3 avec les mêmes règles. Tickets : « Un broker ne répond plus » (1), « La facturation prend du retard, mais nos consommateurs tournent » (2), « Des partitions sont sous-répliquées alors que tous les conteneurs sont `healthy` » (3).
2. **Runbook.** À partir de votre fiche du ticket Paiements, rédigez l'entrée du runbook d'astreinte : symptômes, indicateurs, commandes, correction, validation, et ce qui change en production.
3. **Prévention.** Pour le ticket Stockage, proposez une politique applicable à tous les topics créés par l'équipe Data, et dites comment l'imposer.